<a href="https://colab.research.google.com/github/santiago-zuluaga/Algoritmo-Genetico/blob/main/notebooks/algoritmo_genetico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Google Colab"/></a>

# Laboratorio: algoritmo genético y estrella mágica

En este cuaderno construiremos la idea paso a paso: definiremos el problema, generaremos una población aleatoria, observaremos el fitness y los operadores, y al final seguiremos la evolución generación por generación. El algoritmo y sus operadores se importan del proyecto; aquí nos enfocamos en observarlos.
La secuencia sigue el formato de la clase práctica: fijar parámetros, generar población, definir y comprobar la evaluación, estudiar cada operador y finalmente ejecutar el algoritmo completo.


## 1. ¿Qué es un algoritmo genético?

Es una búsqueda inspirada en la evolución. Tenemos varias soluciones candidatas, medimos qué tan buenas son, elegimos algunas para reproducirse y generamos nuevas candidatas mediante cruce y mutación. Repetimos el proceso en generaciones. En este parcial se minimiza el error de las seis líneas de la estrella. No se garantiza encontrar una solución en toda corrida.
| Término | En este problema |
|---|---|
| **Población** | Conjunto de distribuciones candidatas de los números 1–12. |
| **Individuo** | Una estrella candidata completa. |
| **Cromosoma** | La lista ordenada de 12 genes que representa al individuo. |
| **Gen** | Un número en una posición A–L. |
| **Fitness** | Suma de los errores absolutos de las seis líneas; menor es mejor. |
| **Generación** | Una iteración que reemplaza la población por descendientes y élites. |
| **Selección** | Escoge padres; el torneo favorece fitness bajo. |
| **Cruce** | Combina dos padres; OX mantiene el orden y evita duplicados. |
| **Mutación** | Intercambia dos posiciones para explorar otra distribución. |
| **Elitismo** | Copia los mejores individuos sin cambiarlos a la generación siguiente. |

## 2. Preparar Google Colab y cargar el proyecto

Esta celda clona el repositorio en el directorio temporal de Colab (si todavía no existe), instala las dependencias gráficas y carga los módulos Python del proyecto. No utiliza rutas de Windows.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPO = Path('/content/Algoritmo-Genetico')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/santiago-zuluaga/Algoritmo-Genetico.git', str(REPO)], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib', 'ipywidgets', 'pandas'], check=True)

import random, time
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, Markdown, clear_output, display
import ipywidgets as widgets
from algoritmo_genetico.config import ConfiguracionAG
from algoritmo_genetico.estrella import LINEAS, fitness, sumas_lineas, validar_individuo
from algoritmo_genetico.genetico import (crear_individuo, seleccion_torneo, cruce_ox_detallado, mutacion_swap_detallada, avanzar_poblacion, ejecutar_con_historial)
from algoritmo_genetico.visualizacion import COORDENADAS, ETIQUETAS, dibujar_estrella

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
print('Proyecto listo desde:', REPO)

## 3. Representación de la estrella

El cromosoma usa el orden `[A, B, C, D, E, F, G, H, I, J, K, L]`. Cada posición contiene un número distinto del 1 al 12. Las seis restricciones son:

- **L1** = A + C + F + H = 26
- **L2** = A + D + G + K = 26
- **L3** = B + C + D + E = 26
- **L4** = B + F + I + L = 26
- **L5** = E + G + J + L = 26
- **L6** = H + I + J + K = 26

Para una línea con suma `S`, su error es `abs(S - 26)`. El fitness es la suma de los seis errores. Así, fitness cero significa que las seis sumas valen 26.

In [ ]:
def ver_cromosoma(cromosoma):
    if not validar_individuo(cromosoma):
        raise ValueError('Escribe una permutación exacta de los números del 1 al 12.')
    sumas = sumas_lineas(cromosoma)
    tabla = pd.DataFrame({
        'Línea': [f'L{i}' for i in range(1, 7)],
        'Posiciones': ['A+C+F+H', 'A+D+G+K', 'B+C+D+E', 'B+F+I+L', 'E+G+J+L', 'H+I+J+K'],
        'Suma': sumas, 'Objetivo': [26]*6,
        'Error absoluto': [abs(s-26) for s in sumas]
    })
    print('A–L:', dict(zip(ETIQUETAS, cromosoma)))
    print('Fitness:', fitness(cromosoma))
    display(tabla)
    fig, ax = plt.subplots(figsize=(8, 6))
    dibujar_estrella(ax, cromosoma)
    fig.tight_layout(); plt.show(); plt.close(fig)

entrada = widgets.Text(value='1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12', description='A–L:', layout=widgets.Layout(width='80%'))
boton_ver = widgets.Button(description='Dibujar distribución', button_style='primary')
salida_ver = widgets.Output()
def _dibujar_desde_entrada(_):
    with salida_ver:
        clear_output(wait=True)
        try: ver_cromosoma([int(x.strip()) for x in entrada.value.split(',')])
        except ValueError as error: print(error)
boton_ver.on_click(_dibujar_desde_entrada)
display(widgets.HBox([entrada, boton_ver]), salida_ver)

## 4. Generar la población inicial

Como en el ejemplo de clase, primero fijamos los parámetros y luego generamos la población. Cada individuo se crea mezclando aleatoriamente los números 1–12; no insertamos la solución de referencia. La semilla permite repetir la misma demostración.

In [ ]:
#@title Parámetros de la población para el ejemplo
tamano_demo = 100 #@param {type:"integer"}
semilla_demo = 2026 #@param {type:"integer"}
tamano_demo = max(4, int(tamano_demo))
rng_demo = random.Random(semilla_demo)
poblacion = [crear_individuo(rng_demo) for _ in range(tamano_demo)]
assert all(validar_individuo(ind) for ind in poblacion)

def tabla_poblacion(poblacion, cantidad=5):
    ordenados = sorted(poblacion, key=fitness)[:cantidad]
    return pd.DataFrame([{
        'Individuo': i+1, 'Cromosoma A–L': str(ind), 'Fitness': fitness(ind),
        'Seis sumas': str(sumas_lineas(ind))
    } for i, ind in enumerate(ordenados)])

fitnesses = [fitness(ind) for ind in poblacion]
display(tabla_poblacion(poblacion, 5))
print(f'Población: {len(poblacion)} individuos | mejor fitness: {min(fitnesses)} | promedio: {sum(fitnesses)/len(fitnesses):.2f}')
fig, ax = plt.subplots(figsize=(7, 5)); ax.hist(fitnesses, bins=15, color='#4776a8', edgecolor='white')
ax.set(title='Fitness de la población inicial', xlabel='Fitness (menor es mejor)', ylabel='Cantidad de individuos'); ax.grid(alpha=.2); plt.show(); plt.close(fig)

La tabla muestra cinco individuos con menor fitness, y el histograma deja ver la distribución completa. Que un individuo sea el mejor de esta población no significa que sea una solución: comprobamos las seis sumas, y sólo fitness cero cumple el problema.

## 5. Selección por torneo

El torneo toma una muestra de candidatos y elige el de menor fitness. Aquí se muestran los participantes antes de seleccionar al ganador. Si dos empatan, cualquiera de los empatados puede ganar según el orden del sorteo.

In [ ]:
rng_torneo = random.Random(17)
candidatos = rng_torneo.sample(poblacion, 3)
ganador = min(candidatos, key=fitness)
tabla_torneo = pd.DataFrame([{
    'Cromosoma': str(ind), 'Fitness': fitness(ind),
    'Resultado': 'GANADOR' if ind is ganador else 'participante'
} for ind in candidatos])
display(tabla_torneo)
print('Se selecciona el menor fitness entre estos tres candidatos.')
assert seleccion_torneo(poblacion, 3, random.Random(17)) is ganador

## 6. Cruce Order Crossover (OX)

OX copia un segmento del primer padre en las mismas posiciones del hijo. Luego recorre el segundo padre en orden circular, salta los genes ya copiados y completa los espacios libres. Esto es apropiado para permutaciones porque evita números repetidos y conserva todos los genes.

In [ ]:
padre_1, padre_2 = poblacion[0], poblacion[1]
(hijo_1, hijo_2), (inicio, fin), detalle_ox = cruce_ox_detallado(padre_1, padre_2, random.Random(31))
tabla_ox = pd.DataFrame({
    'Índice': list(range(12)), 'Padre 1': padre_1, 'Padre 2': padre_2,
    'Hijo 1': hijo_1,
    'Parte heredada': ['sí' if inicio <= i <= fin else '' for i in range(12)]
})
display(tabla_ox)
print(f'Segmento heredado de Padre 1: índices {inicio}–{fin} → {detalle_ox["segmento"]}')
print('Orden de relleno desde Padre 2:', detalle_ox['relleno'])
print('Asignaciones (índice, gen):', detalle_ox['pasos'])
print('Hijo válido:', validar_individuo(hijo_1), '| genes únicos:', len(set(hijo_1)))
assert validar_individuo(hijo_1) and validar_individuo(hijo_2)

## 7. Mutación por intercambio

La mutación swap escoge dos posiciones y cambia sus valores. No agrega ni elimina números: el resultado sigue siendo una permutación. Una mutación ocasional ayuda a visitar distribuciones que el cruce de padres actuales podría no producir.

In [ ]:
antes_mutacion = hijo_1.copy()
despues_mutacion, posiciones_mutadas = mutacion_swap_detallada(antes_mutacion, random.Random(8), probabilidad=1.0)
print('Antes:  ', antes_mutacion)
print('Después:', despues_mutacion)
print('Posiciones intercambiadas:', posiciones_mutadas)
print('Fitness antes/después:', fitness(antes_mutacion), fitness(despues_mutacion))
assert validar_individuo(despues_mutacion)

## 8. Elitismo y construcción de una generación

El elitismo copia sin cambios a los mejores individuos. El resto de la generación se completa con padres seleccionados por torneo, cruce OX y mutación swap. El módulo del proyecto realiza esta secuencia en `avanzar_poblacion`.

In [ ]:
config_demo = ConfiguracionAG(tamano_poblacion=100, max_generaciones=100, probabilidad_mutacion=.15, elitismo=2, tamano_torneo=3, semilla=2026)
rng_paso = random.Random(config_demo.semilla)
siguiente = avanzar_poblacion(poblacion, config_demo, rng_paso)
elite_esperada = sorted(poblacion, key=fitness)[:config_demo.elitismo]
print('Élite que se copia:')
display(tabla_poblacion(elite_esperada, config_demo.elitismo))
print('¿Los élites permanecen en la nueva población?', all(ind in siguiente for ind in elite_esperada))
print('Tamaño de la nueva población:', len(siguiente))
assert all(ind in siguiente for ind in elite_esperada)

## 9. Ejecutar y observar la evolución

Ahora usamos el ejecutor real del proyecto, que conserva un resumen por generación: mejor, promedio y peor fitness, diversidad (fracción de cromosomas únicos) y el mejor individuo. Los controles usan `ipywidgets`; **Ejecutar** calcula la corrida, **Siguiente generación** permite inspeccionarla paso a paso y **Animar** reproduce una versión acotada para que no sea pesada.

In [ ]:
control_poblacion = widgets.IntSlider(value=100, min=20, max=200, step=10, description='Población')
control_generaciones = widgets.IntSlider(value=300, min=10, max=1500, step=10, description='Generaciones')
control_mutacion = widgets.FloatSlider(value=.15, min=0, max=.8, step=.05, readout_format='.2f', description='Mutación')
control_elitismo = widgets.IntSlider(value=2, min=0, max=20, step=1, description='Élite')
control_torneo = widgets.IntSlider(value=3, min=2, max=10, step=1, description='Torneo')
control_semilla = widgets.IntText(value=42, description='Semilla')
boton_ejecutar = widgets.Button(description='Ejecutar / reiniciar', button_style='success', icon='play')
boton_siguiente = widgets.Button(description='Siguiente generación', icon='step-forward')
boton_animar = widgets.Button(description='Animar evolución', icon='film')
salida_laboratorio = widgets.Output()
estado = {'corrida': None, 'indice': 0, 'config': None, 'segundos': None}

def _config_desde_controles():
    elite = min(control_elitismo.value, control_poblacion.value - 1)
    torneo = min(control_torneo.value, control_poblacion.value)
    return ConfiguracionAG(control_poblacion.value, control_generaciones.value, control_mutacion.value, elite, torneo, control_semilla.value)

def _mostrar_indice(indice):
    corrida = estado['corrida']; indice = max(0, min(indice, len(corrida.estadisticas)-1))
    estado['indice'] = indice; fila = corrida.estadisticas[indice]
    clear_output(wait=True)
    display(Markdown(f'### Generación {fila.generacion} de {corrida.estadisticas[-1].generacion}\nMejor fitness: **{fila.mejor_fitness}** · promedio: **{fila.fitness_promedio:.2f}** · peor: **{fila.peor_fitness}** · diversidad: **{fila.diversidad:.1%}**'))
    fig, ax = plt.subplots(figsize=(7, 5)); dibujar_estrella(ax, fila.mejor_individuo, f'Mejor de la generación {fila.generacion}')
    fig.tight_layout(); plt.show(); plt.close(fig)
    display(pd.DataFrame({'Línea':[f'L{i}' for i in range(1,7)], 'Suma':sumas_lineas(fila.mejor_individuo), 'Error':[abs(x-26) for x in sumas_lineas(fila.mejor_individuo)]}))

def _ejecutar(_):
    config = _config_desde_controles(); inicio = time.perf_counter()
    corrida = ejecutar_con_historial(config); segundos = time.perf_counter() - inicio
    estado.update(corrida=corrida, indice=0, config=config, segundos=segundos)
    with salida_laboratorio:
        clear_output(wait=True)
        res = corrida.resultado
        print(f'Tiempo: {segundos:.2f} s | Generaciones: {res.generaciones} | Mejor fitness: {res.mejor_fitness}')
        print('Estado:', 'solución encontrada' if res.encontrado else 'no convergió dentro del límite configurado')
        _mostrar_indice(0)
        graficar_historial(corrida)
        resumir_resultado(corrida)

def _siguiente(_):
    with salida_laboratorio:
        if estado['corrida'] is None: print('Primero ejecuta una corrida.'); return
        _mostrar_indice(estado['indice'] + 1)

def _animar(_):
    with salida_laboratorio:
        if estado['corrida'] is None: print('Primero ejecuta una corrida.'); return
        datos = estado['corrida'].estadisticas
        indices = list(range(len(datos)))
        if len(indices) > 80: indices = [round(i*(len(indices)-1)/79) for i in range(80)]
        fig, ax = plt.subplots(figsize=(7, 5))
        def actualizar(k):
            ax.clear(); fila = datos[indices[k]]
            dibujar_estrella(ax, fila.mejor_individuo, f'Generación {fila.generacion} | fitness {fila.mejor_fitness} | promedio {fila.fitness_promedio:.1f}')
            return ax.get_children()
        animacion = FuncAnimation(fig, actualizar, frames=len(indices), interval=350, blit=False)
        plt.close(fig); display(HTML(animacion.to_jshtml(fps=3, default_mode='once')))

boton_ejecutar.on_click(_ejecutar); boton_siguiente.on_click(_siguiente); boton_animar.on_click(_animar)
display(widgets.VBox([control_poblacion, control_generaciones, control_mutacion, control_elitismo, control_torneo, control_semilla, widgets.HBox([boton_ejecutar, boton_siguiente, boton_animar]), salida_laboratorio]))

## 10. Gráficas de convergencia

El mejor y el promedio ayudan a ver si la población mejora; el peor muestra la dispersión de fitness. La diversidad vale entre 0 y 1 y cuenta qué proporción de individuos son cromosomas distintos. Una caída fuerte puede señalar que la población se parece demasiado y explora menos alternativas.

In [ ]:
def graficar_historial(corrida, titulo='Evolución genética'):
    datos = corrida.estadisticas
    generaciones = [x.generacion for x in datos]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(generaciones, [x.mejor_fitness for x in datos], label='Mejor fitness', linewidth=2)
    ax.plot(generaciones, [x.fitness_promedio for x in datos], label='Fitness promedio', alpha=.85)
    ax.plot(generaciones, [x.peor_fitness for x in datos], label='Peor fitness', alpha=.65)
    ax.set(xlabel='Generación', ylabel='Fitness (menor es mejor)', title=titulo); ax.grid(alpha=.25)
    ax2 = ax.twinx(); ax2.plot(generaciones, [x.diversidad for x in datos], color='#8c4bb8', linestyle='--', label='Diversidad')
    ax2.set_ylabel('Fracción de cromosomas distintos'); ax2.set_ylim(0, 1.05)
    lineas, etiquetas = ax.get_legend_handles_labels(); lineas2, etiquetas2 = ax2.get_legend_handles_labels()
    ax.legend(lineas+lineas2, etiquetas+etiquetas2, loc='upper right'); fig.tight_layout(); plt.show(); plt.close(fig)

if estado['corrida'] is not None:
    graficar_historial(estado['corrida'])

### Resultado e interpretación automática

Al terminar, el ejecutor informa si encontró fitness cero o si llegó al máximo de generaciones. El mejor individuo reportado siempre es el mejor observado en la corrida. Una semilla fija reproduce la búsqueda; no garantiza el mismo resultado si cambian los parámetros o el código.

In [ ]:
def resumir_resultado(corrida):
    resultado = corrida.resultado; mejor = resultado.mejor_individuo
    estado_texto = 'encontró una solución válida' if resultado.encontrado else 'no encontró fitness cero antes del límite'
    print(f'El algoritmo {estado_texto}.')
    print('Generaciones ejecutadas:', resultado.generaciones)
    print('Mejor cromosoma A–L:', mejor)
    print('Seis sumas:', sumas_lineas(mejor))
    print('Fitness:', resultado.mejor_fitness)
    if resultado.encontrado:
        print('Las seis restricciones se cumplen: cada línea suma 26.')
    else:
        errores = [abs(s-26) for s in sumas_lineas(mejor)]
        print(f'Quedó una desviación total de {sum(errores)}; errores por línea: {errores}.')

if estado['corrida'] is not None:
    resumir_resultado(estado['corrida']); graficar_historial(estado['corrida'])

## 11. Comparar configuraciones

Comparamos la configuración que elegiste con otra que duplica la probabilidad de mutación (sin superar 1). Es una comparación de parámetros, no una estrategia nueva de diversidad. Ambas corridas usan la misma semilla para facilitar la comparación; los resultados dependen del azar y deben interpretarse como una sola corrida por configuración.

In [ ]:
boton_comparar = widgets.Button(description='Comparar configuración base y exploratoria', button_style='info')
salida_comparacion = widgets.Output()
def _comparar(_):
    with salida_comparacion:
        clear_output(wait=True)
        base = _config_desde_controles()
        exploratoria = ConfiguracionAG(base.tamano_poblacion, base.max_generaciones, min(1.0, base.probabilidad_mutacion*2), base.elitismo, base.tamano_torneo, base.semilla)
        filas=[]; corridas=[]
        for etiqueta, config in [('Base', base), ('Mutación duplicada', exploratoria)]:
            inicio=time.perf_counter(); corrida=ejecutar_con_historial(config); segundos=time.perf_counter()-inicio; corridas.append(corrida)
            filas.append({'Configuración':etiqueta, 'Mutación':config.probabilidad_mutacion, 'Mejor fitness':corrida.resultado.mejor_fitness, 'Encontró solución':corrida.resultado.encontrado, 'Generación final':corrida.resultado.generaciones, 'Tiempo (s)':round(segundos,2), 'Diversidad final':round(corrida.estadisticas[-1].diversidad,3)})
        display(pd.DataFrame(filas))
        fig, ax=plt.subplots(figsize=(9,5))
        for etiqueta, corrida in zip(['Base','Mutación duplicada'],corridas):
            ax.plot([e.generacion for e in corrida.estadisticas],[e.mejor_fitness for e in corrida.estadisticas],label=etiqueta)
        ax.set(xlabel='Generación',ylabel='Mejor fitness',title='Comparación de las corridas'); ax.grid(alpha=.25); ax.legend(); plt.show(); plt.close(fig)
        fig, ax=plt.subplots(figsize=(9,5))
        for etiqueta, corrida in zip(['Base','Mutación duplicada'],corridas):
            ax.plot([e.generacion for e in corrida.estadisticas],[e.diversidad for e in corrida.estadisticas],label=etiqueta)
        ax.set(xlabel='Generación',ylabel='Diversidad',title='Diversidad por configuración'); ax.set_ylim(0,1.05); ax.grid(alpha=.25); ax.legend(); plt.show(); plt.close(fig)
boton_comparar.on_click(_comparar)
display(boton_comparar, salida_comparacion)

## 12. Preguntas para comprobar lo aprendido

1. ¿Por qué cada cromosoma debe ser una permutación de 1 a 12?
2. ¿Qué ventaja tiene OX frente a un cruce que copie genes sin revisar duplicados?
3. Si una línea suma 24, ¿cuánto aporta al fitness?
4. ¿Qué puede pasar con la búsqueda si la diversidad llega casi a cero?
5. ¿Cómo puede ayudar la mutación y qué ocurre si es excesiva?
6. ¿Por qué el elitismo copia algunos individuos sin modificarlos?
7. ¿Por qué el algoritmo no garantiza encontrar fitness cero en un número fijo de generaciones?

## Conclusión

Un algoritmo genético combina selección, cruce, mutación y elitismo para buscar mejores permutaciones. Las estadísticas y la estrella animada permiten observar tanto el avance como el estancamiento; el fitness cero es la comprobación de éxito, no el mero hecho de terminar una corrida.